# Employee ETL into Supabase (Postgres)
Course challenge from an *ETL in Python and SQL* course: load an employee spreadsheet into a Supabase table. The H+ Sport employee spreadsheet is course material and is **not included**, so outputs are cleared.

**Steps**
1. Read the Excel sheet with pandas, drop four columns, save a CSV.
2. Load it with SQLAlchemy + psycopg2 straight into Supabase Postgres. Both attempts failed with `OperationalError` (the direct database host did not accept the connection from Colab).
3. Load it through the Supabase REST API: rename columns to match the table, de-duplicate on `employee_name`, convert types, and post the rows in batches of 200 with `Prefer: resolution=merge-duplicates`. All four batches (200, 200, 200, 141 rows) succeeded.

The Supabase URL and keys are placeholders.

In [ ]:
import pandas as pd
df = pd.read_excel("H+ Sport Employees.xlsx",sheet_name="Employees-Table")

In [ ]:
df.columns

In [ ]:
colmn_to_remove = ["Job Rating","2.91%","New Salary","Tax Rate"]

In [ ]:
Employees = df.drop(colmn_to_remove)

In [ ]:
colmn_to_remove = ["Job Rating","2.91%","New Salary","Tax Rate"]

# Option 1: Create new dataframe
Employees = df.drop(columns=colmn_to_remove)

# Option 2: Modify in place
df.drop(columns=colmn_to_remove, inplace=True)
Employees = df  # if you want to reference it as Employees


In [ ]:
Employees

In [ ]:
df.to_csv("Employees.csv")

In [ ]:
df.head()


In [ ]:
df.tail()

In [ ]:
import pandas as pd

df = pd.read_csv("Employees.csv")

# Rename columns to snake_case
df.columns = [
    "index", "employee_name", "building", "department", "status",
    "hire_date", "month", "years", "benefits", "salary"
]

# Replace NaN with None
df = df.where(pd.notnull(df), None)

# Save cleaned CSV
df.to_csv("employees_clean.csv", index=False)

In [ ]:
!pip install sqlalchemy psycopg2-binary pandas


In [ ]:
%cd /content/course-data
!ls

In [ ]:
from sqlalchemy import create_engine
# from sqlalchemy.pool import NullPool
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

# Fetch variables
USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

# Construct the SQLAlchemy connection string
DATABASE_URL = f"postgresql+psycopg2://<USER>:<PASSWORD>@<HOST>:<PORT>/<DB>"

# Create the SQLAlchemy engine
engine = create_engine(DATABASE_URL)
# If using Transaction Pooler or Session Pooler, we want to ensure we disable SQLAlchemy client side pooling -
# https://docs.sqlalchemy.org/en/20/core/pooling.html#switching-pool-implementations
# engine = create_engine(DATABASE_URL, poolclass=NullPool)

# Test the connection
try:
    with engine.connect() as connection:
        print("Connection successful!")
except Exception as e:
    print(f"Failed to connect: {e}")

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Load CSV
df = pd.read_csv("employees_clean.csv")

# 2. Clean column names to match your table in Supabase
df.columns = [
    "Unnamed: 0", "employee_name", "building", "department", "status",
    "hire_date", "month", "years", "benefits", "salary"
]

# 3. Handle NaN -> None for Postgres compatibility
df = df.where(pd.notnull(df), None)

# 4. Connect to Supabase
DATABASE_URL = "postgresql://<USER>:<PASSWORD>@<HOST>:<PORT>/<DB>"
engine = create_engine(DATABASE_URL)

# 5. Push data into your table
df.to_sql(
    "employees",      # <- must match the table name you created
    engine,
    if_exists="append",  # append instead of replacing
    index=False
)

print("✅ Data uploaded successfully to Supabase!")

In [ ]:
import pandas as pd
import requests
import json
from getpass import getpass

# 1. Load your CSV
df = pd.read_csv("employees_clean.csv")

# Drop any accidental index column from CSV
if "index" in df.columns:
    df = df.drop(columns=["index"])

# # Rename columns to match your Supabase table
df.columns = ["employee_name","building","department","status","hire_date","month","years","benefits","salary"]

df = df.drop_duplicates(subset=["employee_name"])


# Clean types
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
df['hire_date'] = pd.to_datetime(df['hire_date'], errors='coerce').dt.date
df = df.where(pd.notnull(df), None)

# 2. Enter your Supabase credentials (hidden input for security)
# input("Enter Supabase Project URL: ")
SUPABASE_URL = "https://<PROJECT_REF>.supabase.co".strip().rstrip('/')
SUPABASE_SERVICE_ROLE = "<JWT>"
#getpass("Enter Supabase Service Role Key: ")
# 3. Define REST endpoint
TABLE = "employees"
endpoint = f"{SUPABASE_URL}/rest/v1/{TABLE}"

# headers = {
#     "apikey": SUPABASE_SERVICE_ROLE,
#     "Authorization": f"Bearer {SUPABASE_SERVICE_ROLE}",
#     "Content-Type": "application/json",
#     "Prefer": "return=representation"
# }

headers = {
    "apikey": SUPABASE_SERVICE_ROLE,
    "Authorization": f"Bearer {SUPABASE_SERVICE_ROLE}",
    "Content-Type": "application/json",
    "Prefer": "resolution=merge-duplicates"  # <-- automatically update duplicates
}


# ensure date is converted to string before converting to dictionary
df['hire_date'] = df['hire_date'].astype(str)
records = df.to_dict(orient="records")


chunksize = 200
for i in range(0, len(records), chunksize):
    chunk = records[i:i+chunksize]
    resp = requests.post(endpoint, headers=headers, json=chunk)
    if resp.status_code not in (200, 201):
        print("❌ Error:", resp.status_code, resp.text)
        break
    print(f"✅ Inserted {len(chunk)} rows")

In [ ]:
print(df.columns)
# print(len(df.columns))
